<a href="https://colab.research.google.com/github/gausul9365/BUILDING-ML/blob/main/GPT_2_Text_Generation_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



Install Transformers and PyTorch.

Load GPT-2 tokenizer.

Load GPT-2 pretrained weights.

Inspect the model configuration.

Tokenize three different prompts.

Inspect hidden-state and logits shapes.

Generate text using greedy decoding.

Generate text using sampling.
Compare temperature 0.5 and 1.2 with sampling enabled.

Experiment with top-k and top-p.



In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "openai-community/gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(model.config)

prompts = [
    "Artificial intelligence is",
    "A machine learning engineer should",
    "The future of robotics"
]
tokenized_prompts = [tokenizer.encode(prompt, return_tensors="pt") for prompt in prompts]
print(tokenized_prompts)

input_ids = tokenized_prompts[0]
with torch.no_grad():
  outputs = model(input_ids, output_hidden_states=True)


logits = outputs.logits
print(f"Logits shape: {logits.shape}")


hidden_states = outputs.hidden_states
print(f"\nNumber of hidden state layers: {len(hidden_states)}")
print(f"Last hidden state shape: {hidden_states[-1].shape}")
print(f"First hidden state shape: {hidden_states[0].shape}")




Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.22.0",
  "use_cache": true,
  "vocab_size": 50257
}

[tensor([[8001, 9542, 4430,

In [11]:
print("--- Greedy Decoding Generation ---\n")

for i, input_ids in enumerate(tokenized_prompts):

    # Generate text.
    # max_length controls the maximum number of tokens in the final output.
    # We set pad_token_id to avoid a common warning in GPT-2.
    output = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id
    )

    # The output is a tensor of token IDs. We need to decode it back to text.
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"Prompt {i+1}:")
    print(generated_text)
    print("-" * 40)

--- Greedy Decoding Generation ---

Prompt 1:
Artificial intelligence is a new field of research that has been in the works for a while now. It is a field that has been in the works for a while now. It is a field that has been in the works for a while now. It is a field
----------------------------------------
Prompt 2:
A machine learning engineer should be able to predict the future and predict the future.

The problem is that the machine learning problem is not a problem of the future. It is a problem of the past.

The problem is that the future is not a problem of
----------------------------------------
Prompt 3:
The future of robotics is in the hands of the next generation of engineers.

"We're going to see a lot of new technologies coming out of robotics," said Dr. Michael S. Karp, a professor of engineering at the University of California, Berkeley.
----------------------------------------


In [12]:


print("--- Sampling Generation ---\n")

# Setting a manual seed ensures we get the same random results every time we run this cell
torch.manual_seed(42)

for i, input_ids in enumerate(tokenized_prompts):

    # We add do_sample=True to stop greedy decoding and enable probabilistic sampling
    output = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True
    )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"Prompt {i+1}:")
    print(generated_text)
    print("-" * 40)

--- Sampling Generation ---

Prompt 1:
Artificial intelligence is more than just a tool. It has more potential than science to uncover problems and discover new possibilities for humanity. Today's technologies give us insights into life's most profound and immediate dangers. They can provide insight into our world as we know it, and
----------------------------------------
Prompt 2:
A machine learning engineer should consider taking this course. If you have any questions or questions about the course, please speak with our Customer Service representatives. All courses we recommend in this regard are offered in two different English courses: The first is 'Advanced Computer Science in Computer Translation'
----------------------------------------
Prompt 3:
The future of robotics in business, he said, is that there will also be higher standard of service. For instance, it's hard for companies to say no to high-speed transport (like Uber), which can't provide a seamless, service-like exper

In [13]:
import torch

print("--- Temperature Comparison (0.5 vs 1.2) ---\n")

temperatures = [0.5, 1.2]

for temp in temperatures:
    print(f"----TEMPERATURE: {temp}---- ")

    # We set the seed before each temperature run so we can fairly compare how
    # the temperature changes the exact same underlying random state.
    torch.manual_seed(42)

    for i, input_ids in enumerate(tokenized_prompts):

        # Add the temperature parameter
        output = model.generate(
            input_ids,
            max_new_tokens=50,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=temp
        )

        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

        print(f"Prompt {i+1}:")
        print(generated_text)
        print("-" * 40)
    print("\n")

--- Temperature Comparison (0.5 vs 1.2) ---

----TEMPERATURE: 0.5---- 
Prompt 1:
Artificial intelligence is a new field of research that uses algorithms to solve problems, and it's hard to imagine how people can't get a handle on it. One of the most promising areas of research is AI and machine learning.

According to a recent article in
----------------------------------------
Prompt 2:
A machine learning engineer should consider using this tool to learn from a machine learning model, because it's easy to understand how the model works.

In this article, we'll explore how to use this tool to learn from a model to learn from a machine learning model.
----------------------------------------
Prompt 3:
The future of robotics is in the works, and we're going to be creating a whole lot of cool things."

The company's goal is to make robots that can interact with humans and solve problems by using a combination of brain and computer technologies.

"
----------------------------------------


In [14]:
import torch

print("--- Top-K and Top-P Sampling ---\n")

# 1. TOP-K SAMPLING
# top_k=50 means the model only ever chooses from the 50 most likely next words.
print(" ---TOP-K (k=50)---")
torch.manual_seed(42)
for i, input_ids in enumerate(tokenized_prompts):
    output = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50
    )
    print(f"Prompt {i+1}:\n{tokenizer.decode(output[0], skip_special_tokens=True)}\n{'-' * 40}")

# 2. TOP-P (NUCLEUS) SAMPLING
# top_p=0.92 means it chooses from the smallest set of words whose combined probability adds up to 92%.
print("\n========== TOP-P (p=0.92) ==========")
torch.manual_seed(42)
for i, input_ids in enumerate(tokenized_prompts):
    output = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.92,
        top_k=0 # We set top_k to 0 to explicitly disable it and isolate top_p
    )
    print(f"Prompt {i+1}:\n{tokenizer.decode(output[0], skip_special_tokens=True)}\n{'-' * 40}")

--- Top-K and Top-P Sampling ---

 ---TOP-K (k=50)---
Prompt 1:
Artificial intelligence is more than just a tool. It has more potential than science to uncover problems and discover new possibilities for humanity. Today's technologies give us insights into life's most profound and immediate dangers. They can provide insight into our world as we know it, and
----------------------------------------
Prompt 2:
A machine learning engineer should consider taking this course. If you have any questions or questions about the course, please speak with our Customer Service representatives. All courses we recommend in this regard are offered in two different English courses: The first is 'Advanced Computer Science in Computer Translation'
----------------------------------------
Prompt 3:
The future of robotics in business, he said, is that there will also be higher standard of service. For instance, it's hard for companies to say no to high-speed transport (like Uber), which can't provide a sea